# Advanced Clustering: GMM, Spectral Clustering & Evaluation

In this lab you will go beyond K-Means and DBSCAN and explore:

1. **Gaussian Mixture Models (GMM)** — implement the EM algorithm from scratch for soft clustering
2. **Spectral Clustering** — implement clustering via graph Laplacian eigenvectors
3. **Quantitative evaluation** — Adjusted Rand Index, Normalized Mutual Information
4. **Comparison on non-convex datasets** — moons, circles, anisotropic blobs

Prerequisites: you should be comfortable with K-Means and DBSCAN from the previous lab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.stats import multivariate_normal
from scipy.spatial.distance import cdist
np.random.seed(42)

---
## Part A — Gaussian Mixture Models (Expectation-Maximization (EM) Algorithm)

A GMM models data as a mixture of $K$ Gaussian distributions:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

The EM algorithm alternates between:
- **E-step:** compute responsibilities $r_{nk} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_j \pi_j \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$
- **M-step:** update $\pi_k, \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k$ using the responsibilities

### Step A1 — Generate synthetic data

In [ ]:
# Generate 3 well-separated Gaussian blobs
X_gmm, y_gmm = datasets.make_blobs(n_samples=600, centers=3, cluster_std=[1.0, 1.5, 0.8], random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X_gmm[:, 0], X_gmm[:, 1], c=y_gmm, cmap='viridis', s=10, alpha=0.7)
plt.title('Synthetic data — 3 Gaussian blobs'); plt.xlabel('x1'); plt.ylabel('x2')
plt.show()

### Step A2 — Implement the EM algorithm for GMM

Fill in the E-step and M-step.

In [ ]:
def gmm_em(X, K, max_iter=100, tol=1e-6):
    """
    Fit a Gaussian Mixture Model using the EM algorithm.
    
    Parameters:
        X : (N, D) data matrix
        K : number of mixture components
        max_iter : maximum EM iterations
        tol : convergence threshold on log-likelihood change
    
    Returns:
        means : (K, D) cluster means
        covs  : (K, D, D) covariance matrices
        pis   : (K,) mixing coefficients
        responsibilities : (N, K) soft assignments
        log_likelihoods  : list of log-likelihood at each iteration
    """
    N, D = X.shape
    
    # --- Initialization (K-Means init for robustness) ---
    # Random init often gets stuck in local minima.
    # Standard practice: initialize means with K-Means centroids.
    from sklearn.cluster import KMeans
    km_init = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
    means = km_init.cluster_centers_.copy()
    
    # Initialize covariances from K-Means assignments
    covs = np.array([np.cov(X[km_init.labels_ == k].T) + 1e-6 * np.eye(D)
                     if np.sum(km_init.labels_ == k) > D else np.eye(D)
                     for k in range(K)])
    
    # Initialize mixing coefficients from cluster sizes
    pis = np.bincount(km_init.labels_, minlength=K).astype(float) / N
    
    log_likelihoods = []
    
    for iteration in range(max_iter):
        # ========== E-STEP ==========
        # Compute responsibilities r[n, k] 
        # Hint: use scipy.stats.multivariate_normal.pdf(X, mean=mu_k, cov=Sigma_k)
        
        responsibilities = np.zeros((N, K))
        for k in range(K):
            responsibilities[:, k] = ### YOUR CODE HERE ###
        
        # Normalize responsibilities so each row sums to 1
        responsibilities = ### YOUR CODE HERE ###
        
        # ========== COMPUTE LOG-LIKELIHOOD ==========
        # log p(X | params) = sum_n log( sum_k pi_k * N(x_n | mu_k, Sigma_k) )
        ll = 0.0
        for k in range(K):
            ll += pis[k] * multivariate_normal.pdf(X, mean=means[k], cov=covs[k])
        ll = np.sum(np.log(ll + 1e-300))
        log_likelihoods.append(ll)
        
        # Check convergence
        if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
            print(f'Converged at iteration {iteration}')
            break
        
        # ========== M-STEP ==========
        # Effective number of points assigned to cluster k
        N_k = responsibilities.sum(axis=0)  # shape (K,)
        
        for k in range(K):
            # Update mean: mu_k = (1/N_k) * sum_n r_{nk} * x_n
            means[k] = ### YOUR CODE HERE ###
            
            # Update covariance: Sigma_k = (1/N_k) * sum_n r_{nk} * (x_n - mu_k)(x_n - mu_k)^T
            diff = X - means[k]  # (N, D)
            covs[k] = ### YOUR CODE HERE ###
            
            # Add small regularization for numerical stability
            covs[k] += 1e-6 * np.eye(D)
        
        # Update mixing coefficients: pi_k = N_k / N
        pis = ### YOUR CODE HERE ###
    
    return means, covs, pis, responsibilities, log_likelihoods

### Step A3 — Run the EM algorithm and visualize results

In [ ]:
K = 3
means, covs, pis, resp, lls = gmm_em(X_gmm, K, max_iter=200)

# Hard assignments from responsibilities
gmm_labels = np.argmax(resp, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Clustering result
axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1], c=gmm_labels, cmap='viridis', s=10, alpha=0.7)
axes[0].scatter(means[:, 0], means[:, 1], c='red', marker='X', s=200, edgecolors='k')
axes[0].set_title('GMM clustering result')

# 2) Soft assignments (show max responsibility as transparency)
max_resp = resp.max(axis=1)
axes[1].scatter(X_gmm[:, 0], X_gmm[:, 1], c=gmm_labels, cmap='viridis', s=10, alpha=max_resp)
axes[1].set_title('Soft assignments (opacity = max responsibility)')

# 3) Log-likelihood convergence
axes[2].plot(lls, 'b-')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Log-likelihood')
axes[2].set_title('EM convergence')

plt.tight_layout(); plt.show()

print(f'ARI: {adjusted_rand_score(y_gmm, gmm_labels):.4f}')
print(f'NMI: {normalized_mutual_info_score(y_gmm, gmm_labels):.4f}')

### Step A4 — BIC for model selection

The Bayesian Information Criterion helps choose the number of components:

$$\text{BIC} = -2 \ln \hat{L} + p \ln N$$

where $p$ is the number of free parameters and $\hat{L}$ the maximized likelihood.

**Task:** Compute BIC for $K = 1, \ldots, 7$ and pick the best $K$.

In [ ]:
def compute_bic(X, K, log_likelihood):
    """Compute BIC for a GMM with K components."""
    N, D = X.shape
    # Number of free params: K*D (means) + K*D*(D+1)/2 (covs) + (K-1) (pis)
    n_params = K * D + K * D * (D + 1) / 2 + (K - 1)
    return -2 * log_likelihood + n_params * np.log(N)

bic_values = []
K_range = range(1, 8)

for K_test in K_range:
    _, _, _, _, lls_test = gmm_em(X_gmm, K_test, max_iter=200)
    bic_values.append(compute_bic(X_gmm, K_test, lls_test[-1]))
    print(f'K={K_test}, BIC={bic_values[-1]:.1f}')

plt.figure(figsize=(6, 4))
plt.plot(list(K_range), bic_values, 'o-')
plt.xlabel('Number of components K'); plt.ylabel('BIC (lower is better)')
plt.title('BIC for model selection'); plt.show()
print(f'Best K = {list(K_range)[np.argmin(bic_values)]}')

---
## Part B — Spectral Clustering

Spectral clustering works by:
1. Building a **similarity graph** (affinity matrix $W$)
2. Computing the **graph Laplacian** $L = D - W$
3. Finding the bottom-$K$ eigenvectors of $L$ (excluding the trivial all-ones)
4. Running K-Means on the eigenvector embedding

This can cluster **non-convex** shapes that K-Means cannot handle.

### Step B1 — Generate non-convex datasets

In [ ]:
# Generate three challenging datasets
n_samples = 500
X_moons, y_moons = datasets.make_moons(n_samples=n_samples, noise=0.06, random_state=42)
X_circles, y_circles = datasets.make_circles(n_samples=n_samples, noise=0.05, factor=0.5, random_state=42)
X_aniso, y_aniso = datasets.make_blobs(n_samples=n_samples, centers=3, random_state=170)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X_aniso, transformation)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, X_data, y_data, name in zip(axes, 
    [X_moons, X_circles, X_aniso], [y_moons, y_circles, y_aniso],
    ['Two Moons', 'Concentric Circles', 'Anisotropic Blobs']):
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='viridis', s=10)
    ax.set_title(name)
plt.tight_layout(); plt.show()

### Step B2 — Implement Spectral Clustering from scratch

Fill in the missing pieces:
1. Build the RBF (Gaussian) affinity matrix: $W_{ij} = \exp\left(-\frac{\|x_i - x_j\|^2}{2\sigma^2}\right)$
2. Compute the normalized graph Laplacian: $L_{\text{sym}} = I - D^{-1/2} W D^{-1/2}$
3. Find bottom-$K$ eigenvectors and run K-Means on them

In [ ]:
from sklearn.cluster import KMeans

def spectral_clustering(X, n_clusters, sigma=1.0):
    """
    Spectral clustering with RBF affinity and normalized Laplacian.
    
    Parameters:
        X : (N, D) data
        n_clusters : number of clusters
        sigma : bandwidth for the RBF kernel
    Returns:
        labels : (N,) cluster labels
    """
    N = X.shape[0]
    
    # 1. Build affinity matrix W using RBF kernel
    # Hint: use cdist(X, X, 'sqeuclidean') for squared Euclidean distances
    dists_sq = ### YOUR CODE HERE ###
    W = ### YOUR CODE HERE ###
    
    # Set diagonal to 0 (no self-loops)
    np.fill_diagonal(W, 0)
    
    # 2. Compute degree matrix D (diagonal matrix of row sums of W)
    d = ### YOUR CODE HERE ###
    D_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-10))
    
    # 3. Compute normalized Laplacian: L_sym = I - D^{-1/2} W D^{-1/2}
    L_sym = ### YOUR CODE HERE ###
    
    # 4. Eigen-decomposition of L_sym
    eigenvalues, eigenvectors = np.linalg.eigh(L_sym)
    
    # 5. Take the bottom-K eigenvectors (smallest eigenvalues)
    # These correspond to the most connected components of the graph
    U = ### YOUR CODE HERE ###
    
    # 6. Normalize rows of U to unit length (for numerical stability)
    row_norms = np.linalg.norm(U, axis=1, keepdims=True)
    U = U / (row_norms + 1e-10)
    
    # 7. Run K-Means on the eigenvector embedding
    km = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
    labels = km.fit_predict(U)
    
    return labels

### Step B3 — Apply Spectral Clustering to the non-convex datasets

Tune `sigma` for each dataset. A rule of thumb: sigma ≈ median of pairwise distances.

In [ ]:
# Apply spectral clustering
labels_moons   = spectral_clustering(X_moons, 2, sigma=0.15)
labels_circles = spectral_clustering(X_circles, 2, sigma=0.2)
labels_aniso   = spectral_clustering(X_aniso, 3, sigma=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, X_data, labels, y_true, name in zip(axes,
    [X_moons, X_circles, X_aniso],
    [labels_moons, labels_circles, labels_aniso],
    [y_moons, y_circles, y_aniso],
    ['Two Moons', 'Concentric Circles', 'Anisotropic Blobs']):
    ax.scatter(X_data[:, 0], X_data[:, 1], c=labels, cmap='viridis', s=10)
    ari = adjusted_rand_score(y_true, labels)
    ax.set_title(f'{name}\nARI = {ari:.3f}')
plt.suptitle('Spectral Clustering Results', fontsize=14)
plt.tight_layout(); plt.show()

---
## Part C — Grand Comparison

**Task:** Compare K-Means, GMM (sklearn), DBSCAN, and Spectral Clustering on all three datasets.

Evaluate with ARI and Silhouette Score. Which method works best on each dataset and why?

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture

all_datasets = [
    ('Moons',   X_moons,   y_moons,   2),
    ('Circles', X_circles, y_circles, 2),
    ('Aniso',   X_aniso,   y_aniso,   3),
]

methods = {
    'K-Means':   lambda X, k: KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X),
    'GMM':       lambda X, k: GaussianMixture(n_components=k, random_state=42).fit_predict(X),
    'DBSCAN':    lambda X, k: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
    'Spectral':  lambda X, k: SpectralClustering(n_clusters=k, affinity='rbf', gamma=10, random_state=42).fit_predict(X),
}

fig, axes = plt.subplots(len(all_datasets), len(methods), figsize=(18, 12))

for i, (ds_name, X_ds, y_true, n_cl) in enumerate(all_datasets):
    # Scale data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_ds)
    
    for j, (m_name, m_func) in enumerate(methods.items()):
        labels = m_func(X_scaled, n_cl)
        
        # Handle DBSCAN noise label (-1)
        mask = labels >= 0
        ari = adjusted_rand_score(y_true[mask], labels[mask]) if mask.sum() > 0 else 0
        
        axes[i, j].scatter(X_ds[:, 0], X_ds[:, 1], c=labels, cmap='viridis', s=8, alpha=0.7)
        axes[i, j].set_title(f'{m_name}\nARI={ari:.3f}', fontsize=10)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if j == 0:
            axes[i, j].set_ylabel(ds_name, fontsize=12, fontweight='bold')

plt.suptitle('Clustering Methods Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Questions to answer

1. **GMM vs K-Means:** Both converge to the same solution on isotropic Gaussian blobs. When does GMM give different (better) results?
2. **Spectral Clustering:** Why does it succeed on moons and circles where K-Means fails? What is the role of $\sigma$?
3. **DBSCAN:** Which datasets does it handle well? What happens when you vary `eps`?
4. **Soft vs Hard clustering:** Give a real-world example where soft assignments (GMM) would be more useful than hard assignments (K-Means).
5. **Scalability:** Spectral clustering requires computing an $N \times N$ affinity matrix. What is its time complexity and how could you scale it to large datasets?
6. **Bonus:** Implement the **elbow method** for K-Means and compare with BIC for GMM on the blobs dataset.